
# Mueller Matrix U-Net (Notebook)
**Stable HDF5 dataloader, training, and ONNX export**, wired to your path:

```
/Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/hdf5_datasets/hdf5
```

> Start with `num_workers=0` to expose any dataset issues clearly. Once a few batches run, try `num_workers=2`.


In [ ]:

# macOS-safe: use 'spawn' for multiprocessing
def set_start_method_spawn():
    import torch.multiprocessing as mp
    try:
        mp.set_start_method('spawn', force=True)
    except RuntimeError:
        pass  # already set

set_start_method_spawn()
print("Start method set to 'spawn' (or already set).")

import os
from pathlib import Path
from datetime import datetime
import json

import h5py
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Your dataset root
HDF5_DIR = Path("/Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/hdf5_datasets/hdf5")
print("Using HDF5_DIR:", HDF5_DIR)


In [ ]:

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super().__init__()
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)
    def forward(self, x):
        x1 = self.inc(x); x2 = self.down1(x1); x3 = self.down2(x2); x4 = self.down3(x3); x5 = self.down4(x4)
        x = self.up1(x5, x4); x = self.up2(x, x3); x = self.up3(x, x2); x = self.up4(x, x1)
        return self.outc(x)


In [ ]:

def _to_str(x):
    if isinstance(x, (bytes, np.bytes_)):
        return x.decode()
    return str(x)

class MuellerMaskDataset(Dataset):
    def __init__(self, h5_filepath, dataset_type='mueller', target_size=(512, 512), replace_m00_with_nm11s=True):
        self.filepath = os.fspath(h5_filepath)
        self.dataset_type = dataset_type
        self.target_size = target_size
        self.replace_m00 = replace_m00_with_nm11s
        with h5py.File(self.filepath, 'r') as h5f:
            if 'samples' not in h5f:
                raise KeyError("HDF5 is missing group 'samples'")
            self._keys = sorted(h5f['samples'].keys())
            self.num_samples = len(self._keys)
            self.num_classes = int(h5f.attrs.get('num_classes', 2))
            self.hdf5_dataset_type = _to_str(h5f.attrs.get('dataset_type', 'mueller'))
            self.class_labels = [_to_str(v) for v in h5f['class_labels'][:]] if 'class_labels' in h5f else [str(i) for i in range(self.num_classes)]
            self.sample_names = [_to_str(v) for v in h5f['sample_names'][:]] if 'sample_names' in h5f else [k for k in self._keys]
        self.n_channels = 16
        self._h5 = None; self._pid = None

    def _ensure_handle(self):
        pid = os.getpid()
        if (self._h5 is None) or (self._pid != pid):
            if self._h5 is not None:
                try: self._h5.close()
                except Exception: pass
            self._h5 = h5py.File(self.filepath, mode='r', swmr=True, libver='latest')
            self._pid = pid
        return self._h5

    def __len__(self): return self.num_samples

    def _load_from_polarimetric_hdf5(self, h5f, sample_key):
        g = h5f['samples'][sample_key]
        x = g['input'][:]  # (H,W,5)
        if x.ndim != 3 or x.shape[2] < 1:
            raise ValueError(f"{sample_key}: invalid polarimetric shape {x.shape}")
        nM11s = x[:, :, 0]; H, W = nM11s.shape
        nM = np.zeros((H, W, 16), dtype=np.float32)
        nM[:, :, 0] = nM[:, :, 5] = nM[:, :, 10] = nM[:, :, 15] = 1.0
        return nM, nM11s

    def _load_from_mueller_hdf5(self, h5f, sample_key):
        g = h5f['samples'][sample_key]
        nM = g['input'][:]  # (H,W,16)
        if nM.ndim != 3 or nM.shape[2] != 16:
            raise ValueError(f"{sample_key}: expected (H,W,16), got {nM.shape}")
        nM11s = nM[:, :, 0]
        return nM.astype(np.float32, copy=False), nM11s.astype(np.float32, copy=False)

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.num_samples:
            raise IndexError(f"Index {idx} out of range (0..{self.num_samples-1})")
        h5f = self._ensure_handle()
        sample_key = self._keys[idx]
        if sample_key not in h5f['samples']:
            raise KeyError(f"Key '{sample_key}' not found in HDF5 'samples' group.")
        if self.hdf5_dataset_type == 'polarimetric':
            nM, nM11s = self._load_from_polarimetric_hdf5(h5f, sample_key)
        else:
            nM, nM11s = self._load_from_mueller_hdf5(h5f, sample_key)
        if self.replace_m00:
            nM[:, :, 0] = nM11s
        input_data = torch.from_numpy(nM).permute(2, 0, 1).contiguous().float()
        g = h5f['samples'][sample_key]
        if 'target' not in g: raise KeyError(f"{sample_key}: missing 'target' dataset")
        target = torch.from_numpy(g['target'][:]).long()
        if self.target_size is not None:
            input_data = F.interpolate(input_data.unsqueeze(0), size=self.target_size, mode='bilinear', align_corners=True).squeeze(0)
            target = F.interpolate(target.unsqueeze(0).unsqueeze(0).float(), size=self.target_size, mode='nearest').squeeze(0).squeeze(0).long()
        t_min = int(target.min().item()); t_max = int(target.max().item())
        if t_min < 0 or t_max >= self.num_classes:
            raise ValueError(f"{sample_key}: target labels [{t_min},{t_max}] outside 0..{self.num_classes-1}")
        name = _to_str(g.attrs.get('sample_name', sample_key))
        return {'input': input_data, 'target': target, 'name': name}


In [ ]:

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        pred = torch.softmax(pred, dim=1)
        tgt1h = torch.nn.functional.one_hot(target, num_classes=pred.shape[1]).permute(0,3,1,2).float()
        inter = (pred * tgt1h).sum(dim=(2,3))
        union = pred.sum(dim=(2,3)) + tgt1h.sum(dim=(2,3))
        dice = (2.*inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

class CombinedLoss(nn.Module):
    def __init__(self, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss()
        self.wce = ce_weight; self.wdice = dice_weight
    def forward(self, pred, target):
        return self.wce * self.ce(pred, target) + self.wdice * self.dice(pred, target)


In [ ]:

def build_dataloaders(train_dataset, val_dataset, batch_size=4, num_workers=0):
    use_workers = max(0, int(num_workers))
    common = dict(batch_size=batch_size, pin_memory=True, num_workers=use_workers, persistent_workers=(use_workers>0))
    if use_workers > 0: common['prefetch_factor'] = 2
    train_loader = DataLoader(train_dataset, shuffle=True, **common)
    val_loader   = DataLoader(val_dataset,   shuffle=False, **common)
    return train_loader, val_loader


In [ ]:

def train_unet_model(
    train_h5_path,
    val_h5_path,
    input_type='mueller',
    target_size=(512, 512),
    batch_size=4,
    num_epochs=50,
    learning_rate=1e-4,
    device='cuda',
    save_dir='./models',
    num_workers=0
):
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    dev = torch.device(device if torch.cuda.is_available() else 'cpu')
    print("Using device:", dev)

    print("Loading datasets...")
    train_dataset = MuellerMaskDataset(train_h5_path, input_type, target_size)
    val_dataset   = MuellerMaskDataset(val_h5_path,   input_type, target_size)
    train_loader, val_loader = build_dataloaders(train_dataset, val_dataset, batch_size=batch_size, num_workers=num_workers)

    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Input channels: {train_dataset.n_channels}")
    print(f"Number of classes: {train_dataset.num_classes} -> {train_dataset.class_labels}")

    model = UNet(train_dataset.n_channels, train_dataset.num_classes, bilinear=True).to(dev)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    criterion = CombinedLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5)

    def _train_epoch():
        model.train(); total=0.0
        for batch in tqdm(train_loader, desc="Training"):
            x = batch['input'].to(dev, non_blocking=True)
            y = batch['target'].to(dev, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total += loss.item()
        return total / max(1, len(train_loader))

    @torch.no_grad()
    def _validate():
        model.eval(); total=0.0; correct=0; pixels=0
        for batch in tqdm(val_loader, desc="Validation"):
            x = batch['input'].to(dev, non_blocking=True)
            y = batch['target'].to(dev, non_blocking=True)
            out = model(x)
            loss = criterion(out, y)
            total += loss.item()
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            pixels  += y.numel()
        return total / max(1, len(val_loader)), (correct / pixels) if pixels else 0.0

    best_val = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    print(f"Starting training for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        tr = _train_epoch()
        vl, va = _validate()
        scheduler.step(vl)
        history['train_loss'].append(tr); history['val_loss'].append(vl); history['val_accuracy'].append(va)
        print(f"Train Loss: {tr:.4f}")
        print(f"Val   Loss: {vl:.4f} | Acc: {va:.4f}")
        if vl < best_val:
            best_val = vl
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': vl,
                'val_accuracy': va,
                'config': {
                    'input_type': input_type,
                    'n_channels': train_dataset.n_channels,
                    'n_classes': train_dataset.num_classes,
                    'target_size': target_size
                }
            }, save_dir / 'best_model.pth')
            print(f"Saved best model (val_loss={vl:.4f})")
    torch.save(model.state_dict(), save_dir / 'final_model.pth')
    with open(save_dir / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    return model, history


In [ ]:

def export_to_onnx(model_path, onnx_path, input_type='mueller', target_size=(512,512), opset_version=12):
    checkpoint = torch.load(model_path, map_location='cpu')
    cfg = checkpoint['config']
    model = UNet(cfg['n_channels'], cfg['n_classes'], bilinear=True)
    model.load_state_dict(checkpoint['model_state_dict']); model.eval()
    n_channels = 16 if input_type == 'mueller' else 1
    dummy = torch.randn(1, n_channels, target_size[0], target_size[1])
    torch.onnx.export(model, dummy, onnx_path, export_params=True, opset_version=opset_version,
                      do_constant_folding=True, input_names=['input'], output_names=['output'],
                      dynamic_axes={{'input': {{0: 'batch_size'}}, 'output': {{0: 'batch_size'}}}})
    print("Model exported to ONNX:", onnx_path)
    meta = {'input_type': input_type, 'n_channels': n_channels, 'n_classes': cfg['n_classes'],
            'target_size': target_size, 'created': datetime.now().isoformat()}
    meta_path = Path(onnx_path).with_suffix('.json')
    import json
    with open(meta_path, 'w') as f: json.dump(meta, f, indent=2)
    print("Metadata saved to:", meta_path)


In [ ]:

# --- Optional: quick debug to surface dataset issues without training ---
try:
    dbg_ds = MuellerMaskDataset(HDF5_DIR / "mueller/train.h5", target_size=(512,512))
    print("Samples:", len(dbg_ds), "Classes:", dbg_ds.num_classes, dbg_ds.class_labels[:8])
    x = dbg_ds[0]['input']; y = dbg_ds[0]['target']
    print("One sample -> x:", tuple(x.shape), "y:", tuple(y.shape), "x dtype:", x.dtype, "y dtype:", y.dtype)
except Exception as e:
    print("Dataset quick check failed:", e)


In [ ]:

# === Example run (uncomment to train) ===
model, history = train_unet_model(
    train_h5_path=HDF5_DIR / "mueller/train.h5",
    val_h5_path=HDF5_DIR / "mueller/validation.h5",
    input_type='mueller',
    target_size=(512, 512),
    batch_size=4,
    num_epochs=5,
    learning_rate=1e-4,
    device='cuda',
    save_dir='./models/mueller_unet',
    num_workers=0
)
export_to_onnx(
    model_path='./models/mueller_unet/best_model.pth',
    onnx_path='./models/mueller_unet/mueller_unet.onnx',
    input_type='mueller',
    target_size=(512, 512)
)
print("Notebook wired to your path. Un-comment the block above to train and export.")
